In [ ]:
import json

import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats

from emu_renewal.constants import DATA_PATH, FULL_RUN
from emu_renewal.outputs import get_param_vals_by_analysis, get_ratios_from_disps, get_median_ratios
from emu_renewal.plotting import plot_dispersion_ratio_figure
from emu_renewal.utils import get_analysis_paths, get_country_name

set_matplotlib_formats("svg")

In [ ]:
SEED = 42
ANALYSIS = "oxcgrt_floored"
BASELINE = "no_scaling"
N_EXEMPLARS = 3

all_countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json"))
analysis_paths = get_analysis_paths(FULL_RUN, all_countries)
avail_countries = [
    iso3 for iso3, paths in analysis_paths.items() if BASELINE in paths and ANALYSIS in paths
]

disp_posts = {}
for iso3 in avail_countries:
    c_paths = {k: analysis_paths[iso3][k] for k in (BASELINE, ANALYSIS)}
    disp_posts[iso3] = get_param_vals_by_analysis("dispersion_proc", c_paths)

np.random.seed(SEED)
ratio_dists = get_ratios_from_disps(disp_posts)
median_ratios = get_median_ratios(ratio_dists, ANALYSIS)

In [ ]:
# One country drawn at random from each tertile of the median ratio
rng = np.random.default_rng(SEED)
ratio_series = pd.Series(median_ratios).sort_values()
tertiles = pd.qcut(ratio_series, N_EXEMPLARS, labels=False)

exemplar_iso3s = []
for tier in range(N_EXEMPLARS):
    pool = ratio_series.index[tertiles == tier].tolist()
    exemplar_iso3s.append(str(rng.choice(pool)))
exemplar_iso3s = list(reversed(exemplar_iso3s))

exemplar_df = pd.DataFrame(
    {
        "iso3": exemplar_iso3s,
        "country": [get_country_name(c) for c in exemplar_iso3s],
        "median_ratio": [median_ratios[c] for c in exemplar_iso3s],
        "tertile": list(reversed(range(1, N_EXEMPLARS + 1))),
    }
)

In [ ]:
fig = plot_dispersion_ratio_figure(
    disp_posts,
    ratio_dists,
    exemplar_iso3s,
    analysis_type=ANALYSIS,
    baseline=BASELINE,
)
display(fig)